In [ ]:
import cv2
import os
import torch
from ultralytics import YOLO
from ultralytics.engine.results import Results
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel

# 0. SEGÉDFÜGGVÉNY: SAHI -> YOLO konvertáló
def sahi_to_yolo_plot(frame, sahi_results, category_mapping, hide_labels=False, hide_conf=False):
    """
    Átveszi a SAHI találatait, és becsomagolja egy YOLO objektumba, 
    így a rajzolást a YOLO gyári, megszokott színpalettájával tudjuk elvégezni!
    """
    boxes = []
    for obj in sahi_results.object_prediction_list:
        # A dobozhoz hozzávesszük a tulajdonságokat
        boxes.append([
            obj.bbox.minx, obj.bbox.miny, obj.bbox.maxx, obj.bbox.maxy,
            obj.score.value, obj.category.id
        ])

    # Ha van találat, tenzort csinálunk belőle, ha nincs, üres tenzort
    if len(boxes) > 0:
        boxes = torch.tensor(boxes)
    else:
        boxes = torch.empty((0, 6))

    # SAHI kategóriák ID-jainak egységesítése
    names = {int(k): v for k, v in category_mapping.items()}
    yolo_res = Results(orig_img=frame, path='', names=names, boxes=boxes)

    # Rárajzoljuk a képre
    return yolo_res.plot(labels=not hide_labels, conf=not hide_conf)

# Képek feldolgozása

In [ ]:
# 1. Kép feldolgozása YOLO felismeréssel

def create_annotated_image(input_path, output_path, model_path='best.pt', conf_threshold=0.35, hide_labels=False, hide_conf=False):
    model = YOLO(model_path)
    
    frame = cv2.imread(input_path)
    if frame is None:
        print(f"Hiba: Nem tudtam megnyitni a képet: {input_path}")
        return
            
    print(f"Bemeneti kép (Sima): {input_path}")
    
    results = model(
        source=frame,
        conf=conf_threshold,
        iou=0.4,
        agnostic_nms=True,
        verbose=False
    )
        
    # Rajzolás a YOLO motorral
    annotated_frame = results[0].plot(labels=not hide_labels, conf=not hide_conf)
        
    cv2.imwrite(output_path, annotated_frame)
    print(f"Az új kép elmentve: {output_path}\n")

In [ ]:

# 2. Kép feldolgozása SAHI szeleteléssel

def create_annotated_image_slicing(input_path, output_path, model_path='best.pt', conf_threshold=0.32, tile_size=512, overlap_ratio=0.2, hide_labels=False, hide_conf=False):
    if not os.path.exists(input_path):
        print(f"Hiba: Nem tudtam megnyitni a képet: {input_path}")
        return

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='yolov8',
        model_path=model_path,
        confidence_threshold=conf_threshold,
        device="cuda:0" if torch.cuda.is_available() else "cpu"
    )
    
    frame = cv2.imread(input_path)
    print(f"Bemeneti kép (Slicing): {input_path}")

    results = get_sliced_prediction(
        input_path,
        detection_model,
        slice_height=tile_size,
        slice_width=tile_size,
        overlap_height_ratio=overlap_ratio,
        overlap_width_ratio=overlap_ratio,
        postprocess_type="NMS",
        postprocess_match_threshold=0.4,
        postprocess_class_agnostic=True,
        verbose=False
    )

    # Rajzolás a Segédfüggvényünkkel
    annotated_frame = sahi_to_yolo_plot(frame, results, detection_model.category_mapping, hide_labels, hide_conf)
    
    cv2.imwrite(output_path, annotated_frame)
    print(f"Az új szeletelt kép elmentve: {output_path}\n")

## A következő cellában lehet képeket beadni futtatásra

### Használata:
- Az /imgs mappában lévő kép nevét átírni egy egész számra, png formátumban (pl. 5.png), a kódban az i változót átírni a kapott számra
- A kép lefut mind a YOLO, mind a SAHI szeletelő függvénnyel is.
- A YOLO kimenete ai.png, a SAHi kimenete ai_s.png végződésű
- A kimenet az /imgs/outputs mappában valósul meg

In [ ]:
if __name__ == "__main__":
    i=2
    img_input_path=f"imgs/{i}.jpg"
    img_output_path=f"imgs/outputs/{i}ai.png"
    create_annotated_image(img_input_path, img_output_path, conf_threshold=0.24, hide_conf=True)

    img_output_path_s=f"imgs/outputs/{i}ai_s.png"
    create_annotated_image_slicing(img_input_path, img_output_path_s, conf_threshold=0.36, hide_conf=True, tile_size=480, hide_labels=False)

# Videók feldolgozása

In [ ]:
# 1. Videó feldolgozása YOLO felismeréssel

def create_annotated_video(input_path, output_path, model_path='best.pt', conf_threshold=0.35, hide_labels=False, hide_conf=False):
    model = YOLO(model_path)

    video_capture = cv2.VideoCapture(input_path)
    video_capture.set(cv2.CAP_PROP_ORIENTATION_AUTO, 1.0)

    if not video_capture.isOpened():
        print(f"Hiba: Nem tudtam megnyitni a videót: {input_path}")
        return

    success, first_frame = video_capture.read()
    if not success: return
        
    frame_height, frame_width = first_frame.shape[:2]
    fps = int(video_capture.get(cv2.CAP_PROP_FPS)) or 30

    print(f"Bemeneti videó (Sima): {input_path}")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    frame = first_frame
    while success:
        results = model.track(
            source=frame, 
            conf=conf_threshold, 
            iou=0.4,             
            agnostic_nms=True,   
            persist=True,
            tracker="botsort.yaml",
            verbose=False
        )

        # ID-k eltüntetése a trackerből
        if results[0].boxes is not None and results[0].boxes.is_track:
            results[0].boxes.data = results[0].boxes.data[:, [0, 1, 2, 3, 5, 6]]

        # Rajzolás a YOLO motorral
        annotated_frame = results[0].plot(labels=not hide_labels, conf=not hide_conf)
        video_writer.write(annotated_frame)
            
        success, frame = video_capture.read()

    video_capture.release()
    video_writer.release()
    print(f"Sikeresen befejezve! Az új videó elmentve: {output_path}\n")


In [ ]:
# 2. Videó feldolgozása SAHI szeleteléssel

def create_annotated_video_slicing(input_path, output_path, model_path='best.pt', conf_threshold=0.32, tile_size=512, overlap_ratio=0.2, hide_labels=False, hide_conf=False):
    if not os.path.exists(input_path):
        print(f"Hiba: Nem tudtam megnyitni a videót: {input_path}")
        return

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='yolov8',
        model_path=model_path,
        confidence_threshold=conf_threshold
    )

    video_capture = cv2.VideoCapture(input_path)
    video_capture.set(cv2.CAP_PROP_ORIENTATION_AUTO, 1.0)

    success, first_frame = video_capture.read()
    if not success: return

    frame_height, frame_width = first_frame.shape[:2]
    fps = int(video_capture.get(cv2.CAP_PROP_FPS)) or 30

    print(f"Bemeneti videó (Slicing): {input_path}")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    frame = first_frame
    while success:
        results = get_sliced_prediction(
            frame,
            detection_model,
            slice_height=tile_size,
            slice_width=tile_size,
            overlap_height_ratio=overlap_ratio,
            overlap_width_ratio=overlap_ratio,
            postprocess_type="NMS",
            postprocess_match_threshold=0.4,
            postprocess_class_agnostic=True,
            verbose=False
        )

        # Rajzolás a Segédfüggvényünkkel
        annotated_frame = sahi_to_yolo_plot(frame, results, detection_model.category_mapping, hide_labels, hide_conf)
        video_writer.write(annotated_frame)
        
        success, frame = video_capture.read()

    video_capture.release()
    video_writer.release()
    print(f"Sikeresen befejezve! Az új szeletelt videó elmentve: {output_path}\n")

## A következő cellában lehet videókat beadni futtatásra

### Használata:
- Az /videos mappában lévő vidseó nevét átírni egy egész számra, mp4 formátumban (pl. 5.mp4), a kódban az i változót átírni a kapott számra
- A kép lefut mind a YOLO, mind a SAHI szeletelő függvénnyel is.
- A YOLO kimenete ai.mp4, a SAHi kimenete ai_s.mp4 végződésű
- A kimenet az /imgs/outputs mappában valósul meg

In [ ]:
if __name__ == "__main__":
    i=1
    input_path = f"videos/t{i}.mp4"
    output_path = f"videos/outputs/t{i}ai.mp4"
    #create_annotated_video(input_path, output_path, conf_threshold=0.37)

    output_path_s = f"videos/outputs/t{i}ai_s.mp4"
    #create_annotated_video_slicing(input_path, output_path_s, conf_threshold=0.37, tile_size=640)